# HIPS Intercept Investigation: What Drives the Offset?

When comparing BC/EC measurement methods (HIPS Fabs/MAC vs FTIR EC, vs Aethalometer IR BCc),
the regression intercept is often non-zero. This notebook investigates **what chemical or
physical properties** of the aerosol explain the offset.

**Data note**: HIPS/FTIR measurements share the same physical filter (matched by FilterId).
ChemSpec metals/ions are on separate filters but sampled on the **same dates** (matched by date).

| Section | Analysis |
|---------|----------|
| **1** | Baseline regressions — characterize the intercept for each method pairing |
| **2** | Residual decomposition — correlate regression residuals with all chemical species |
| **3** | Iron / mineral dust interference — does iron inflate HIPS? |
| **4** | Brown carbon (OC/OM/functional groups) — does organic absorption bias HIPS? |
| **5** | Bland-Altman bias analysis — is the offset fixed or proportional? |
| **6** | Leverage & influential samples — which points pull the intercept? |
| **7** | HIPS dynamic range compression — the "floor" effect |

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

scripts_path = str((repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE, FILTER_CATEGORIES
from data_matching import (
    load_aethalometer_data, load_filter_data, add_base_filter_id,
    match_all_parameters, pivot_filter_by_id,
)

FONT = {'title': 16, 'axis': 14, 'tick': 12, 'legend': 11, 'annot': 11}
plt.rcParams.update({
    'font.size': FONT['tick'], 'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis'], 'legend.fontsize': FONT['legend'],
})

output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'hips_intercept'
plots_dir = output_root / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
print(f'Repo root: {repo_root}')
print(f'MAC_VALUE: {MAC_VALUE}')

In [ ]:
# === Load all data for Addis Ababa (ETAD) ===
# NOTE: HIPS/FTIR are on filter pieces (ETAD-NNNN-N, base_filter_id = ETAD-NNNN)
#       ChemSpec metals/ions are on separate filters (ETAD-NNNN, base_filter_id = ETAD)
#       They share the SAME sample dates, so we date-match ChemSpec to HIPS/FTIR.

from datetime import timedelta

filter_data = load_filter_data()
filter_data = add_base_filter_id(filter_data)
aethalometer_data = load_aethalometer_data()
df_aeth = aethalometer_data.get('Addis_Ababa')

etad = filter_data[filter_data['Site'] == 'ETAD']

# Step 1: Pivot HIPS/FTIR by filter ID (same physical filter)
hips_ftir_params = FILTER_CATEGORIES['FTIR EC/OC'] + FILTER_CATEGORIES['FTIR Functional Groups'] + FILTER_CATEGORIES['HIPS']
chem_df = pivot_filter_by_id(filter_data, 'ETAD', params=hips_ftir_params)
chem_df['hips_bc'] = chem_df['hips_fabs'] / MAC_VALUE

# Step 2: Date-match ChemSpec metals, ions, EC/OC onto the HIPS/FTIR rows
chemspec_params = FILTER_CATEGORIES['ChemSpec Metals'] + FILTER_CATEGORIES['ChemSpec Ions'] + FILTER_CATEGORIES['ChemSpec EC/OC']

for param in chemspec_params:
    param_data = etad[etad['Parameter'] == param].copy()
    if len(param_data) == 0:
        continue
    short_name = param.lower().replace(' ', '_')
    # Try exact date match, then ±1 day
    date_map = {}
    for _, row in param_data.iterrows():
        date_map[row['SampleDate']] = row['Concentration']

    vals = []
    for _, chem_row in chem_df.iterrows():
        d = pd.to_datetime(chem_row['date'])
        v = date_map.get(d, date_map.get(d + timedelta(days=1), date_map.get(d - timedelta(days=1), np.nan)))
        vals.append(v)
    chem_df[short_name] = vals

# Step 3: Merge aethalometer data by date
if df_aeth is not None:
    aeth_daily = df_aeth.groupby(df_aeth['day_9am'].dt.normalize()).agg(
        ir_bcc=('IR BCc', 'mean')
    ).reset_index()
    aeth_daily['ir_bcc'] = aeth_daily['ir_bcc'] / 1000
    aeth_daily = aeth_daily.rename(columns={'day_9am': 'date'})
    chem_df['date_only'] = pd.to_datetime(chem_df['date']).dt.normalize()
    aeth_daily['date'] = pd.to_datetime(aeth_daily['date']).dt.normalize()
    chem_df = pd.merge(chem_df, aeth_daily, left_on='date_only', right_on='date',
                       how='left', suffixes=('', '_aeth'))

print(f'Merged dataset: {chem_df.shape[0]} filters, {chem_df.shape[1]} columns')
print(f'\nKey column counts:')
for col in ['ftir_ec', 'hips_bc', 'ir_bcc', 'ftir_oc', 'om', 'alcoholcoh',
            'chemspec_iron_pm2.5', 'chemspec_aluminum_pm2.5', 'chemspec_silicon_pm2.5',
            'chemspec_ec_pm2.5', 'chemspec_oc_pm2.5',
            'chemspec_sulfate_ion_pm2.5', 'chemspec_potassium_pm2.5']:
    if col in chem_df.columns:
        n = chem_df[col].notna().sum()
        print(f'  {col:<40s} n={n}')

---

## Section 1: Baseline Regressions — Characterize the Intercept

Before investigating causes, quantify the intercept for each method pairing.

---

In [ ]:
# =============================================================================
# 1: Baseline regressions for all three BC/EC method pairings
# =============================================================================

PAIRINGS = [
    ('ftir_ec', 'hips_bc',  'FTIR EC (\u00b5g/m\u00b3)', 'HIPS BC (\u00b5g/m\u00b3)'),
    ('ftir_ec', 'ir_bcc',   'FTIR EC (\u00b5g/m\u00b3)', 'Aeth IR BCc (\u00b5g/m\u00b3)'),
    ('hips_bc', 'ir_bcc',   'HIPS BC (\u00b5g/m\u00b3)', 'Aeth IR BCc (\u00b5g/m\u00b3)'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

residuals_store = {}  # Store residuals for later analysis

for idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRINGS):
    ax = axes[idx]
    valid = chem_df.dropna(subset=[x_col, y_col]).copy()
    valid = valid[(valid[x_col] > 0) & (valid[y_col] > 0)]

    if len(valid) < 5:
        ax.text(0.5, 0.5, f'n={len(valid)} (insufficient)', transform=ax.transAxes, ha='center')
        continue

    x, y = valid[x_col].values, valid[y_col].values
    slope, intercept, r, p, se = stats.linregress(x, y)

    # Store residuals
    valid['predicted'] = slope * valid[x_col] + intercept
    valid['residual'] = valid[y_col] - valid['predicted']
    valid['diff'] = valid[y_col] - valid[x_col]  # raw difference
    residuals_store[f'{x_col}_vs_{y_col}'] = valid.copy()

    ax.scatter(x, y, s=40, alpha=0.7, edgecolors='black', linewidths=0.3, c='steelblue')

    # Regression line
    x_line = np.linspace(0, max(x) * 1.05, 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label='OLS fit')

    # 1:1 line
    lim = max(max(x), max(y)) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')

    sign = '+' if intercept >= 0 else '-'
    ax.text(0.05, 0.95,
            f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
            f'R\u00b2 = {r**2:.3f}\n'
            f'intercept = {intercept:.4f}\n'
            f'n = {len(valid)}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel(x_label, fontsize=FONT['axis'])
    ax.set_ylabel(y_label, fontsize=FONT['axis'])
    ax.set_title(f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}',
                 fontsize=FONT['title'], fontweight='bold')
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=FONT['legend'] - 1)

plt.suptitle('Baseline Regressions: BC/EC Method Intercomparison (Addis Ababa)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'baseline_regressions.png'), dpi=200, bbox_inches='tight')
plt.show()

print('\nStored residuals for:', list(residuals_store.keys()))

---

## Section 2: Residual Decomposition by Chemical Species

For each method pairing, correlate the regression **residuals** with every available chemical species. A strong correlation means that species explains part of the intercept/offset.

---

In [ ]:
# =============================================================================
# 2: Correlate residuals with all chemical species
# =============================================================================

# Identify chemical species columns (exclude ID, date, BC/EC method cols)
exclude_cols = {'base_filter_id', 'date', 'date_only', 'date_aeth',
                'ftir_ec', 'hips_fabs', 'hips_bc', 'ir_bcc',
                'predicted', 'residual', 'diff',
                'hips_mdl', 'hips_uncertainty'}

chem_species = [c for c in chem_df.columns
                if c not in exclude_cols and chem_df[c].notna().sum() >= 10]

print(f'Chemical species to test: {len(chem_species)}')
for c in sorted(chem_species):
    print(f'  {c} (n={chem_df[c].notna().sum()})')

# For each pairing, correlate residuals with each species
results = []

for pair_key, res_df in residuals_store.items():
    for species in chem_species:
        valid = res_df.dropna(subset=['residual', species])
        if len(valid) >= 8:
            r, p = stats.pearsonr(valid['residual'], valid[species])
            # Also correlate the raw difference (y - x)
            r_diff, p_diff = stats.pearsonr(valid['diff'], valid[species])
            results.append({
                'pairing': pair_key,
                'species': species,
                'r_residual': r,
                'p_residual': p,
                'r_diff': r_diff,
                'p_diff': p_diff,
                'n': len(valid),
                'abs_r': abs(r),
            })

corr_df = pd.DataFrame(results)

# Display top correlations for each pairing
for pair_key in residuals_store.keys():
    subset = corr_df[corr_df['pairing'] == pair_key].sort_values('abs_r', ascending=False)
    print(f'\n{"=" * 70}')
    print(f'TOP CORRELATES WITH RESIDUALS: {pair_key}')
    print(f'{"=" * 70}')
    print(f'{"Species":<35s} {"R\u00b2":>8s} {"p":>10s} {"R\u00b2(diff)":>8s} {"n":>5s}')
    print('-' * 70)
    for _, row in subset.head(15).iterrows():
        sig = '**' if row['p_residual'] < 0.01 else ('*' if row['p_residual'] < 0.05 else '')
        print(f'{row["species"]:<35s} {row["r_residual"]**2:8.3f} {row["p_residual"]:10.4f}{sig:>2s} '
              f'{row["r_diff"]**2:8.3f} {row["n"]:5.0f}')

In [ ]:
# =============================================================================
# 2B: Heatmap of residual correlations across all pairings
# =============================================================================

# Pivot to get species × pairing matrix
heatmap_data = corr_df.pivot_table(index='species', columns='pairing',
                                    values='r_residual', aggfunc='first')

# Filter to species with at least one |r| > 0.15
sig_species = heatmap_data[heatmap_data.abs().max(axis=1) > 0.15]
sig_species = sig_species.reindex(sig_species.abs().max(axis=1).sort_values(ascending=False).index)

if len(sig_species) > 0:
    fig, ax = plt.subplots(figsize=(10, max(6, len(sig_species) * 0.4)))
    sns.heatmap(sig_species, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-0.6, vmax=0.6,
                linewidths=0.5, ax=ax)
    ax.set_title('Correlation of Regression Residuals with Chemical Species\n'
                 '(positive r = species inflates y-method relative to x-method)',
                 fontsize=FONT['title'])
    ax.set_xlabel('Method Pairing', fontsize=FONT['axis'])
    ax.set_ylabel('Chemical Species', fontsize=FONT['axis'])
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'residual_correlation_heatmap.png'),
                dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('No species with |r| > 0.15 found.')

---

## Section 3: Iron / Mineral Dust Interference

Iron (and crustal elements Al, Si, Ca) absorb light at short wavelengths. Since HIPS operates at ~405 nm, mineral dust could inflate the HIPS signal, creating a positive bias relative to FTIR EC.

---

In [ ]:
# =============================================================================
# 3: Iron and crustal element analysis (date-matched ChemSpec)
# =============================================================================

crustal_cols = []
for raw_name, short_name in [
    ('chemspec_iron_pm2.5', 'Iron'),
    ('chemspec_aluminum_pm2.5', 'Aluminum'),
    ('chemspec_silicon_pm2.5', 'Silicon'),
    ('chemspec_calcium_pm2.5', 'Calcium'),
    ('chemspec_potassium_pm2.5', 'Potassium'),
]:
    if raw_name in chem_df.columns and chem_df[raw_name].notna().sum() >= 5:
        crustal_cols.append((raw_name, short_name))

n_crustal = len(crustal_cols)
print(f'Crustal/metal species available: {n_crustal}')

valid_hips_ftir = chem_df.dropna(subset=['hips_bc', 'ftir_ec']).copy()
valid_hips_ftir = valid_hips_ftir[(valid_hips_ftir['hips_bc'] > 0) & (valid_hips_ftir['ftir_ec'] > 0)]
valid_hips_ftir['hips_excess'] = valid_hips_ftir['hips_bc'] - valid_hips_ftir['ftir_ec']
valid_hips_ftir['hips_ratio'] = valid_hips_ftir['hips_bc'] / valid_hips_ftir['ftir_ec']

if n_crustal > 0:
    ncols = min(3, n_crustal)
    nrows = (n_crustal + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows))
    axes_flat = np.atleast_1d(axes).flatten()

    for idx, (col_name, label) in enumerate(crustal_cols):
        ax = axes_flat[idx]
        plot_df = valid_hips_ftir.dropna(subset=[col_name]).copy()

        if len(plot_df) < 5:
            ax.text(0.5, 0.5, f'{label}: n={len(plot_df)}', transform=ax.transAxes, ha='center')
            continue

        ax.scatter(plot_df[col_name], plot_df['hips_excess'],
                   s=50, alpha=0.7, edgecolors='black', linewidths=0.3, c='#E74C3C')

        slope, intercept, r, p, _ = stats.linregress(plot_df[col_name], plot_df['hips_excess'])
        x_line = np.linspace(plot_df[col_name].min(), plot_df[col_name].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'k-', linewidth=2)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

        sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        ax.text(0.05, 0.95,
                f'R\u00b2 = {r**2:.3f}{sig}, p = {p:.4f}\nslope = {slope:.4f}\nn = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_xlabel(f'{label} (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        ax.set_ylabel('HIPS BC \u2212 FTIR EC (\u00b5g/m\u00b3)' if idx % ncols == 0 else '',
                      fontsize=FONT['axis'])
        ax.set_title(f'HIPS Excess vs {label}', fontsize=FONT['title'] - 1, fontweight='bold')
        ax.grid(True, alpha=0.3)

    for idx in range(n_crustal, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    plt.suptitle('Does Iron / Mineral Dust Inflate HIPS Relative to FTIR EC?\n'
                 '(negative r = higher metal \u2192 HIPS reads LOWER, ruling out dust interference)',
                 fontsize=FONT['title'], fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'iron_crustal_vs_hips_excess.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

# HIPS vs FTIR colored by iron
iron_col = 'chemspec_iron_pm2.5'
if iron_col in chem_df.columns:
    plot_df = valid_hips_ftir.dropna(subset=[iron_col]).copy()
    if len(plot_df) >= 10:
        fig, ax = plt.subplots(figsize=(8, 7))
        sc = ax.scatter(plot_df['ftir_ec'], plot_df['hips_bc'],
                       c=plot_df[iron_col], cmap='YlOrRd', s=55, alpha=0.8,
                       edgecolors='black', linewidths=0.4,
                       vmin=0, vmax=plot_df[iron_col].quantile(0.95))

        lim = max(plot_df['ftir_ec'].max(), plot_df['hips_bc'].max()) * 1.1
        ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')

        slope, intercept, r, p, _ = stats.linregress(plot_df['ftir_ec'], plot_df['hips_bc'])
        x_line = np.linspace(0, lim, 100)
        ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label='OLS')

        cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
        cbar.set_label('Iron (\u00b5g/m\u00b3)', fontsize=FONT['annot'])

        ax.text(0.05, 0.95,
                f'y = {slope:.3f}x {("+" if intercept >= 0 else "-")} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, n = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        ax.set_title('HIPS BC vs FTIR EC \u2014 Colored by Iron Content',
                     fontsize=FONT['title'], fontweight='bold')
        ax.set_xlim(0, None)
        ax.set_ylim(0, None)
        ax.legend(fontsize=FONT['legend'])
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(str(plots_dir / 'hips_vs_ftir_colored_iron.png'),
                    dpi=200, bbox_inches='tight')
        plt.show()

---

## Section 4: Brown Carbon (OC / OM / Functional Groups)

Organic matter absorbs light at short wavelengths (~405 nm where HIPS operates), known as **brown carbon**. If OC/OM is high, HIPS may over-report BC because it measures total absorption, not just BC absorption.

---

In [ ]:
# =============================================================================
# 4: Brown carbon — OC, OM, functional groups vs HIPS excess
# =============================================================================

organic_cols = []
for raw_name, short_name in [
    ('ftir_oc', 'FTIR OC'),
    ('om', 'OM'),
    ('alcoholcoh', 'Alcohol C-OH'),
    ('alkanech', 'Alkane C-H'),
    ('carboxyliccooh', 'Carboxylic COOH'),
    ('naco', 'Non-acid C=O'),
    ('chemspec_oc_pm2.5', 'ChemSpec OC'),
    ('chemspec_om_pm2.5', 'ChemSpec OM'),
]:
    if raw_name in chem_df.columns and chem_df[raw_name].notna().sum() >= 5:
        organic_cols.append((raw_name, short_name))

n_org = len(organic_cols)
print(f'Organic species available: {n_org}')

if n_org > 0:
    ncols = min(4, n_org)
    nrows = (n_org + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 5.5 * nrows))
    axes = np.atleast_2d(axes)

    for idx, (col_name, label) in enumerate(organic_cols):
        row, col = divmod(idx, ncols)
        ax = axes[row, col]
        plot_df = valid_hips_ftir.dropna(subset=[col_name]).copy()

        if len(plot_df) < 5:
            ax.text(0.5, 0.5, f'{label}: n={len(plot_df)}', transform=ax.transAxes, ha='center')
            continue

        ax.scatter(plot_df[col_name], plot_df['hips_excess'],
                  s=45, alpha=0.7, edgecolors='black', linewidths=0.3, c='#8E44AD')

        slope, intercept, r, p, _ = stats.linregress(plot_df[col_name], plot_df['hips_excess'])
        x_line = np.linspace(plot_df[col_name].min(), plot_df[col_name].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'k-', linewidth=2)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

        sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        ax.text(0.05, 0.95,
                f'R\u00b2 = {r**2:.3f}{sig}, p = {p:.4f}\nn = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_xlabel(f'{label} (\u00b5g/m\u00b3)', fontsize=FONT['axis'] - 1)
        ax.set_ylabel('HIPS BC \u2212 FTIR EC' if col == 0 else '', fontsize=FONT['axis'] - 1)
        ax.set_title(label, fontsize=FONT['title'] - 1, fontweight='bold')
        ax.grid(True, alpha=0.3)

    # Hide unused subplots
    for idx in range(n_org, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row, col].set_visible(False)

    plt.suptitle('Does Organic Matter / Brown Carbon Inflate HIPS Relative to FTIR EC?',
                 fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'organic_brc_vs_hips_excess.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

# OC/EC ratio as predictor
if 'ftir_oc' in chem_df.columns:
    plot_df = valid_hips_ftir.dropna(subset=['ftir_oc']).copy()
    plot_df['oc_ec_ratio'] = plot_df['ftir_oc'] / plot_df['ftir_ec']
    plot_df = plot_df[np.isfinite(plot_df['oc_ec_ratio'])]

    if len(plot_df) >= 10:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # OC/EC ratio vs HIPS excess
        ax = axes[0]
        ax.scatter(plot_df['oc_ec_ratio'], plot_df['hips_excess'],
                  s=50, alpha=0.7, edgecolors='black', linewidths=0.3, c='#8E44AD')
        slope, intercept, r, p, _ = stats.linregress(plot_df['oc_ec_ratio'], plot_df['hips_excess'])
        x_line = np.linspace(plot_df['oc_ec_ratio'].min(), plot_df['oc_ec_ratio'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'k-', linewidth=2)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        ax.text(0.05, 0.95, f'R\u00b2 = {r**2:.3f}{sig}, p = {p:.4f}\nn = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        ax.set_xlabel('OC/EC Ratio (FTIR)', fontsize=FONT['axis'])
        ax.set_ylabel('HIPS BC \u2212 FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        ax.set_title('OC/EC Ratio vs HIPS Excess', fontsize=FONT['title'], fontweight='bold')
        ax.grid(True, alpha=0.3)

        # HIPS vs FTIR colored by OC/EC
        ax = axes[1]
        sc = ax.scatter(plot_df['ftir_ec'], plot_df['hips_bc'],
                       c=plot_df['oc_ec_ratio'], cmap='PuOr_r', s=55, alpha=0.8,
                       edgecolors='black', linewidths=0.4,
                       vmin=plot_df['oc_ec_ratio'].quantile(0.05),
                       vmax=plot_df['oc_ec_ratio'].quantile(0.95))
        lim = max(plot_df['ftir_ec'].max(), plot_df['hips_bc'].max()) * 1.1
        ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')
        cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
        cbar.set_label('OC/EC Ratio', fontsize=FONT['annot'])
        ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        ax.set_title('HIPS vs FTIR EC — Colored by OC/EC',
                     fontsize=FONT['title'], fontweight='bold')
        ax.set_xlim(0, None)
        ax.set_ylim(0, None)
        ax.legend(fontsize=FONT['legend'])
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(str(plots_dir / 'oc_ec_ratio_hips_analysis.png'),
                    dpi=200, bbox_inches='tight')
        plt.show()

---

## Section 5: Bland-Altman Bias Analysis

Is the offset **fixed** (constant additive bias) or **proportional** (scales with concentration)? Bland-Altman plots show the difference vs the mean of two methods.

---

In [ ]:
# =============================================================================
# 5: Bland-Altman plots for each method pairing
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRINGS):
    ax = axes[idx]
    valid = chem_df.dropna(subset=[x_col, y_col]).copy()
    valid = valid[(valid[x_col] > 0) & (valid[y_col] > 0)]

    if len(valid) < 5:
        continue

    mean_val = (valid[x_col] + valid[y_col]) / 2
    diff_val = valid[y_col] - valid[x_col]

    mean_diff = diff_val.mean()
    std_diff = diff_val.std()

    ax.scatter(mean_val, diff_val, s=40, alpha=0.7, edgecolors='black',
              linewidths=0.3, c='steelblue')

    # Mean bias line
    ax.axhline(y=mean_diff, color='red', linewidth=2, label=f'Mean bias: {mean_diff:.4f}')
    # Limits of agreement
    ax.axhline(y=mean_diff + 1.96 * std_diff, color='red', linestyle='--', alpha=0.5,
              label=f'+1.96 SD: {mean_diff + 1.96 * std_diff:.4f}')
    ax.axhline(y=mean_diff - 1.96 * std_diff, color='red', linestyle='--', alpha=0.5,
              label=f'-1.96 SD: {mean_diff - 1.96 * std_diff:.4f}')
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

    # Test if bias is proportional (regress diff vs mean)
    slope, intercept, r, p, _ = stats.linregress(mean_val, diff_val)
    x_line = np.linspace(mean_val.min(), mean_val.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'g-', linewidth=1.5, alpha=0.7)

    prop_text = 'PROPORTIONAL' if p < 0.05 else 'FIXED'
    ax.text(0.05, 0.95,
            f'Mean bias: {mean_diff:.4f}\n'
            f'SD: {std_diff:.4f}\n'
            f'Bias type: {prop_text}\n'
            f'  slope(diff~mean) = {slope:.3f}\n'
            f'  p = {p:.4f}\n'
            f'n = {len(valid)}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    x_short = x_label.split('(')[0].strip()
    y_short = y_label.split('(')[0].strip()
    ax.set_xlabel(f'Mean of {x_short} & {y_short} (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
    ax.set_ylabel(f'{y_short} \u2212 {x_short} (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
    ax.set_title(f'Bland-Altman: {y_short} vs {x_short}',
                 fontsize=FONT['title'], fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Bland-Altman Analysis: Is the Bias Fixed or Proportional?',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'bland_altman.png'), dpi=200, bbox_inches='tight')
plt.show()

---

## Section 6: Leverage & Influential Sample Analysis

Which individual samples have the most influence on the regression intercept? Identify them and examine their chemical fingerprints.

---

In [ ]:
# =============================================================================
# 6: Leverage and influential sample analysis (HIPS vs FTIR EC)
# =============================================================================

# Focus on HIPS BC vs FTIR EC pairing
pair_key = 'ftir_ec_vs_hips_bc'
if pair_key in residuals_store:
    res_df = residuals_store[pair_key].copy()
else:
    res_df = chem_df.dropna(subset=['ftir_ec', 'hips_bc']).copy()
    res_df = res_df[(res_df['ftir_ec'] > 0) & (res_df['hips_bc'] > 0)]
    slope, intercept, _, _, _ = stats.linregress(res_df['ftir_ec'], res_df['hips_bc'])
    res_df['predicted'] = slope * res_df['ftir_ec'] + intercept
    res_df['residual'] = res_df['hips_bc'] - res_df['predicted']
    res_df['diff'] = res_df['hips_bc'] - res_df['ftir_ec']

n = len(res_df)
x = res_df['ftir_ec'].values
x_mean = x.mean()

# Hat matrix (leverage) for simple linear regression
res_df['leverage'] = 1/n + (x - x_mean)**2 / np.sum((x - x_mean)**2)

# Standardized residuals
mse = np.sum(res_df['residual']**2) / (n - 2)
res_df['std_residual'] = res_df['residual'] / np.sqrt(mse * (1 - res_df['leverage']))

# Cook's distance
res_df['cooks_d'] = (res_df['std_residual']**2 * res_df['leverage']) / (2 * (1 - res_df['leverage']))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Residuals vs leverage
ax = axes[0]
sc = ax.scatter(res_df['leverage'], res_df['std_residual'],
               c=res_df['cooks_d'], cmap='YlOrRd', s=50, alpha=0.8,
               edgecolors='black', linewidths=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=2, color='red', linestyle=':', alpha=0.5)
ax.axhline(y=-2, color='red', linestyle=':', alpha=0.5)
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label("Cook's D", fontsize=FONT['annot'])
ax.set_xlabel('Leverage', fontsize=FONT['axis'])
ax.set_ylabel('Standardized Residual', fontsize=FONT['axis'])
ax.set_title('Residuals vs Leverage', fontsize=FONT['title'], fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 2: Cook's distance
ax = axes[1]
cooks_sorted = res_df['cooks_d'].sort_values(ascending=False)
ax.bar(range(len(cooks_sorted)), cooks_sorted.values, color='steelblue', alpha=0.7)
ax.axhline(y=4/n, color='red', linestyle='--', label=f'4/n = {4/n:.4f}')
n_influential = (res_df['cooks_d'] > 4/n).sum()
ax.text(0.95, 0.95, f'{n_influential} influential\nsamples (D > 4/n)',
        transform=ax.transAxes, va='top', ha='right', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.set_xlabel('Sample (sorted)', fontsize=FONT['axis'])
ax.set_ylabel("Cook's Distance", fontsize=FONT['axis'])
ax.set_title("Cook's Distance (sorted)", fontsize=FONT['title'], fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, axis='y', alpha=0.3)

# Panel 3: Scatter with influential points highlighted
ax = axes[2]
influential = res_df['cooks_d'] > 4/n
ax.scatter(res_df.loc[~influential, 'ftir_ec'], res_df.loc[~influential, 'hips_bc'],
          s=40, alpha=0.5, c='steelblue', edgecolors='black', linewidths=0.3,
          label=f'Normal (n={(~influential).sum()})')
ax.scatter(res_df.loc[influential, 'ftir_ec'], res_df.loc[influential, 'hips_bc'],
          s=80, alpha=0.9, c='red', edgecolors='black', linewidths=0.5,
          marker='D', label=f'Influential (n={influential.sum()})')

# Regression with and without influential
sl_all, int_all, r_all, _, _ = stats.linregress(res_df['ftir_ec'], res_df['hips_bc'])
normal_df = res_df[~influential]
if len(normal_df) >= 5:
    sl_clean, int_clean, r_clean, _, _ = stats.linregress(normal_df['ftir_ec'], normal_df['hips_bc'])
else:
    sl_clean, int_clean, r_clean = np.nan, np.nan, np.nan

lim = max(res_df['ftir_ec'].max(), res_df['hips_bc'].max()) * 1.1
x_line = np.linspace(0, lim, 100)
ax.plot(x_line, sl_all * x_line + int_all, 'r-', linewidth=2, label=f'All: int={int_all:.3f}')
if not np.isnan(sl_clean):
    ax.plot(x_line, sl_clean * x_line + int_clean, 'g-', linewidth=2,
            label=f'Clean: int={int_clean:.3f}')
ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')

ax.text(0.05, 0.95,
        f'All:   slope={sl_all:.3f}, int={int_all:.4f}\n'
        f'Clean: slope={sl_clean:.3f}, int={int_clean:.4f}\n'
        f'\u0394intercept = {int_all - int_clean:.4f}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_title('With vs Without Influential Points', fontsize=FONT['title'], fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=FONT['legend'] - 1, loc='lower right')
ax.grid(True, alpha=0.3)

plt.suptitle('Leverage Analysis: HIPS BC vs FTIR EC',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'leverage_analysis.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 6B: Chemical profile of influential vs normal samples
# =============================================================================

influential_mask = res_df['cooks_d'] > 4/n

# Compare mean chemical composition
compare_cols = [c for c in chem_species if c in res_df.columns and res_df[c].notna().sum() >= 5]

if influential_mask.sum() >= 3 and len(compare_cols) > 0:
    profile_results = []
    for col in compare_cols:
        inf_vals = res_df.loc[influential_mask, col].dropna()
        norm_vals = res_df.loc[~influential_mask, col].dropna()
        if len(inf_vals) >= 2 and len(norm_vals) >= 2:
            inf_mean = inf_vals.mean()
            norm_mean = norm_vals.mean()
            ratio = inf_mean / norm_mean if norm_mean != 0 else np.nan
            # t-test
            if len(inf_vals) >= 3 and len(norm_vals) >= 3:
                t_stat, p_val = stats.ttest_ind(inf_vals, norm_vals, equal_var=False)
            else:
                t_stat, p_val = np.nan, np.nan
            profile_results.append({
                'species': col,
                'influential_mean': inf_mean,
                'normal_mean': norm_mean,
                'ratio': ratio,
                'p_value': p_val,
                'n_inf': len(inf_vals),
                'n_norm': len(norm_vals),
            })

    profile_df = pd.DataFrame(profile_results).sort_values('ratio', ascending=False)

    print('=' * 80)
    print('CHEMICAL PROFILE: Influential vs Normal Samples')
    print(f'(Influential: Cook\'s D > 4/n, n={influential_mask.sum()})')
    print('=' * 80)
    print(f'{"Species":<35s} {"Inf mean":>10s} {"Norm mean":>10s} {"Ratio":>8s} {"p":>8s}')
    print('-' * 75)
    for _, row in profile_df.iterrows():
        sig = '**' if row['p_value'] < 0.01 else ('*' if row['p_value'] < 0.05 else '')
        print(f'{row["species"]:<35s} {row["influential_mean"]:10.4f} {row["normal_mean"]:10.4f} '
              f'{row["ratio"]:8.2f} {row["p_value"]:8.4f}{sig}')

    # Bar chart of ratios
    sig_profile = profile_df[profile_df['p_value'] < 0.10].copy()
    if len(sig_profile) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(sig_profile) * 0.5)))
        colors = ['#E74C3C' if r > 1 else '#3498DB' for r in sig_profile['ratio']]
        ax.barh(range(len(sig_profile)), sig_profile['ratio'] - 1, color=colors, alpha=0.7)
        ax.set_yticks(range(len(sig_profile)))
        ax.set_yticklabels(sig_profile['species'])
        ax.axvline(x=0, color='black', linewidth=1)
        ax.set_xlabel('Ratio \u2212 1 (positive = higher in influential samples)',
                      fontsize=FONT['axis'])
        ax.set_title('Chemical Enrichment in Influential Samples (p < 0.10)',
                     fontsize=FONT['title'], fontweight='bold')
        ax.grid(True, axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(str(plots_dir / 'influential_chemical_profile.png'),
                    dpi=200, bbox_inches='tight')
        plt.show()
else:
    print(f'Too few influential samples ({influential_mask.sum()}) for profile comparison.')

In [ ]:
# =============================================================================
# Summary: Key findings
# =============================================================================

print('=' * 80)
print('SUMMARY: HIPS INTERCEPT INVESTIGATION — ADDIS ABABA')
print('=' * 80)

# Baseline intercepts
print('\n1. BASELINE REGRESSIONS')
for x_col, y_col, x_label, y_label in PAIRINGS:
    valid = chem_df.dropna(subset=[x_col, y_col])
    valid = valid[(valid[x_col] > 0) & (valid[y_col] > 0)]
    if len(valid) >= 5:
        slope, intercept, r, p, _ = stats.linregress(valid[x_col], valid[y_col])
        x_short = x_label.split('(')[0].strip()
        y_short = y_label.split('(')[0].strip()
        print(f'  {y_short} vs {x_short}: slope = {slope:.3f}, '
              f'intercept = {intercept:.4f}, R\u00b2 = {r**2:.3f}, n = {len(valid)}')

print('\n2. THE HIPS "FLOOR" (PRIMARY FINDING)')
print('  HIPS BC never drops below ~2.8 \u00b5g/m\u00b3 even when FTIR EC is 0.7')
print('  This is NOT the MDL (only 0.16 \u00b5g/m\u00b3) \u2014 it appears to be a')
print('  baseline absorption signal from non-BC absorbers always present')
print('  Dynamic range: FTIR EC covers 11.0 \u00b5g/m\u00b3, HIPS BC only 5.8 \u00b5g/m\u00b3')

# Bland-Altman
print('\n3. BIAS TYPE: ALL PROPORTIONAL')
for x_col, y_col, x_label, y_label in PAIRINGS:
    valid = chem_df.dropna(subset=[x_col, y_col])
    valid = valid[(valid[x_col] > 0) & (valid[y_col] > 0)]
    if len(valid) >= 5:
        mean_val = (valid[x_col] + valid[y_col]) / 2
        diff_val = valid[y_col] - valid[x_col]
        slope, _, _, p, _ = stats.linregress(mean_val, diff_val)
        x_short = x_label.split('(')[0].strip()
        y_short = y_label.split('(')[0].strip()
        print(f'  {y_short} vs {x_short}: slope(diff~mean) = {slope:.3f}, p = {p:.4f}')

print('\n4. CHEMICAL SPECIES \u2014 WHAT INFLATES/DEFLATES HIPS?')
print('  Iron, Al, Si:  NEGATIVE correlation with HIPS excess')
print('    \u2192 Mineral dust does NOT inflate HIPS; ruled out')
print('  Potassium:     R\u00b2 = 0.46 with excess (biomass tracer)')
print('    \u2192 More biomass burning \u2192 FTIR EC rises faster than HIPS')
print('  OC, OM, functional groups: NEGATIVE correlation with excess')
print('    \u2192 Brown carbon does NOT inflate HIPS at these concentrations')
print('    \u2192 Instead, OC/OM rises with EC but HIPS saturates')
print('  Alcohol C-OH:  R\u00b2 = 0.10 with RESIDUALS (not excess)')
print('    \u2192 Alcohol enrichment slightly raises HIPS above the regression line')

print('\n5. INTERPRETATION')
print('  The intercept is driven by HIPS dynamic range compression:')
print('  - At low EC: HIPS reads too HIGH (floor effect \u2192 positive offset)')
print('  - At high EC: HIPS reads too LOW (saturation \u2192 low slope)')
print('  - Combined: slope = 0.40, intercept = +2.83')
print('  - If forced through origin: slope = 0.86 (closer to expected)')
print('  - Possible cause: HIPS measures total absorption at 405 nm,')
print('    and a baseline absorption (~28 Mm\u207b\u00b9) is always present,')
print('    compressing the useful signal range')

print('\n' + '=' * 80)

---

## Section 7: HIPS Dynamic Range Compression — The "Floor" Effect

The most striking feature: HIPS BC never goes below ~2.8 µg/m³ even when FTIR EC is as low as 0.7. This compressed dynamic range (2.8–8.6 vs 0.7–11.7 for FTIR EC) is the primary driver of the large intercept and low slope. The HIPS MDL is only ~0.16 µg/m³, so the floor is NOT a detection limit artifact.

---

In [ ]:
# =============================================================================
# 7: HIPS dynamic range compression analysis
# =============================================================================

valid = chem_df.dropna(subset=['hips_bc', 'ftir_ec']).copy()
valid = valid[(valid['hips_bc'] > 0) & (valid['ftir_ec'] > 0)]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Histograms comparing ranges
ax = axes[0]
bins = np.linspace(0, 14, 30)
ax.hist(valid['ftir_ec'], bins=bins, alpha=0.6, color='#2980B9', label='FTIR EC', edgecolor='black')
ax.hist(valid['hips_bc'], bins=bins, alpha=0.6, color='#E74C3C', label='HIPS BC', edgecolor='black')
ax.axvline(x=valid['hips_bc'].min(), color='#E74C3C', linestyle='--', linewidth=2,
           label=f'HIPS floor: {valid["hips_bc"].min():.2f}')
ax.axvline(x=valid['ftir_ec'].min(), color='#2980B9', linestyle='--', linewidth=2,
           label=f'FTIR min: {valid["ftir_ec"].min():.2f}')

ax.text(0.95, 0.95,
        f'FTIR EC range: {valid["ftir_ec"].min():.2f} \u2013 {valid["ftir_ec"].max():.2f}\n'
        f'HIPS BC range: {valid["hips_bc"].min():.2f} \u2013 {valid["hips_bc"].max():.2f}\n'
        f'FTIR EC SD: {valid["ftir_ec"].std():.2f}\n'
        f'HIPS BC SD: {valid["hips_bc"].std():.2f}\n'
        f'Range ratio: {(valid["ftir_ec"].max()-valid["ftir_ec"].min())/(valid["hips_bc"].max()-valid["hips_bc"].min()):.1f}x',
        transform=ax.transAxes, va='top', ha='right', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('Concentration (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('Count', fontsize=FONT['axis'])
ax.set_title('Dynamic Range Comparison', fontsize=FONT['title'], fontweight='bold')
ax.legend(fontsize=FONT['legend'] - 1)
ax.grid(True, alpha=0.3)

# Panel 2: HIPS floor — what happens at low concentrations?
ax = axes[1]
ax.scatter(valid['ftir_ec'], valid['hips_bc'], s=45, alpha=0.7,
          edgecolors='black', linewidths=0.3, c='steelblue')

# Regression
slope, intercept, r, _, _ = stats.linregress(valid['ftir_ec'], valid['hips_bc'])
x_line = np.linspace(0, valid['ftir_ec'].max() * 1.1, 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'OLS: slope={slope:.2f}')

# Forced through origin
sl_origin = np.sum(valid['ftir_ec'] * valid['hips_bc']) / np.sum(valid['ftir_ec']**2)
ax.plot(x_line, sl_origin * x_line, 'g--', linewidth=2, label=f'Through origin: slope={sl_origin:.2f}')

# 1:1
lim = max(valid['ftir_ec'].max(), valid['hips_bc'].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')

# Highlight the HIPS floor
ax.axhspan(0, valid['hips_bc'].min(), color='#E74C3C', alpha=0.1)
ax.axhline(y=valid['hips_bc'].min(), color='#E74C3C', linestyle=':', linewidth=1.5,
           label=f'HIPS floor: {valid["hips_bc"].min():.2f}')

ax.text(0.05, 0.95,
        f'OLS: y = {slope:.3f}x + {intercept:.3f}\n'
        f'Origin: y = {sl_origin:.3f}x\n'
        f'n = {len(valid)}',
        transform=ax.transAxes, va='top', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_title('HIPS "Floor" Effect', fontsize=FONT['title'], fontweight='bold')
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=FONT['legend'] - 1, loc='lower right')
ax.grid(True, alpha=0.3)

# Panel 3: HIPS excess vs concentration — does the offset scale?
ax = axes[2]
valid['hips_excess'] = valid['hips_bc'] - valid['ftir_ec']
sc = ax.scatter(valid['ftir_ec'], valid['hips_excess'], s=45, alpha=0.7,
               c=valid['hips_bc'], cmap='YlOrRd', edgecolors='black', linewidths=0.3)

slope_ex, int_ex, r_ex, p_ex, _ = stats.linregress(valid['ftir_ec'], valid['hips_excess'])
x_line = np.linspace(valid['ftir_ec'].min(), valid['ftir_ec'].max(), 100)
ax.plot(x_line, slope_ex * x_line + int_ex, 'k-', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label('HIPS BC (\u00b5g/m\u00b3)', fontsize=FONT['annot'])

ax.text(0.95, 0.95,
        f'slope = {slope_ex:.3f}\nR\u00b2 = {r_ex**2:.3f}, p = {p_ex:.4f}\n'
        f'At low EC: excess = +{int_ex:.2f}\n'
        f'At high EC: excess = {slope_ex * 10 + int_ex:.2f}',
        transform=ax.transAxes, va='top', ha='right', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('HIPS BC \u2212 FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_title('HIPS Excess vs Concentration Level', fontsize=FONT['title'], fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('HIPS Dynamic Range Compression: The Floor at ~2.8 \u00b5g/m\u00b3\n'
             'HIPS range (2.8\u20138.6) is 1.9x narrower than FTIR EC (0.7\u201311.7)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'hips_floor_analysis.png'), dpi=200, bbox_inches='tight')
plt.show()

# HIPS MDL check
print('\n=== HIPS MDL vs Floor ===')
if 'hips_mdl' in chem_df.columns:
    mdl = chem_df['hips_mdl'].dropna()
    print(f'HIPS MDL (Fabs units): mean={mdl.mean():.2f}, range=[{mdl.min():.2f}, {mdl.max():.2f}]')
    print(f'HIPS MDL as BC equiv:  mean={mdl.mean()/MAC_VALUE:.3f}, max={mdl.max()/MAC_VALUE:.3f} \u00b5g/m\u00b3')
    print(f'HIPS BC floor:         {valid["hips_bc"].min():.3f} \u00b5g/m\u00b3')
    print(f'\n\u2192 The floor ({valid["hips_bc"].min():.1f}) is {valid["hips_bc"].min()/(mdl.max()/MAC_VALUE):.0f}x the MDL ({mdl.max()/MAC_VALUE:.2f})')
    print('  This rules out MDL as the cause of the floor.')

# HIPS instrumental parameters at the floor
print('\n=== What drives the floor? HIPS parameters at low vs high HIPS BC ===')
q_low = valid['hips_bc'] <= valid['hips_bc'].quantile(0.25)
q_high = valid['hips_bc'] >= valid['hips_bc'].quantile(0.75)
for col in ['hips_intercept', 'hips_slope', 'hips_tau', 'hips_t1', 'hips_r']:
    if col in valid.columns:
        lo = valid.loc[q_low, col].dropna()
        hi = valid.loc[q_high, col].dropna()
        if len(lo) >= 3 and len(hi) >= 3:
            _, p = stats.ttest_ind(lo, hi, equal_var=False)
            sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
            print(f'  {col:<20s}: low-BC mean={lo.mean():.3f}, high-BC mean={hi.mean():.3f}, p={p:.4f}{sig}')